In [64]:
import numpy as np
import pandas as pd
import sklearn.linear_model as skl_lm
import matplotlib.pyplot as plt
import sklearn.neighbors as skl_nb
import sklearn.preprocessing as skl_pre
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report


In [65]:
csv_path = 'training_data.csv'
training_data = pd.read_csv(csv_path)
np.random.seed(1)

trainI = np.random.choice(training_data.shape[0], size=1280, replace=False)
trainIndex = training_data.index.isin(trainI)
train = training_data.iloc[trainIndex]  # training set
test = training_data.iloc[~trainIndex]  # test set

# Var classification
X_train = train.drop(columns=['increase_stock'])
Y_train = train['increase_stock']
X_test = test.drop(columns=['increase_stock'])
Y_test = test['increase_stock']

labels = np.unique(Y_test)
print(labels)

numeric_var = ['temp', 'dew', 'humidity', 'precip', 'snowdepth', 'windspeed', 'cloudcover', 'visibility']
categorical_var = ['hour_of_day', 'day_of_week', 'month']
binomial_var = ['holiday', 'weekday', 'summertime', 'snow']

['high_bike_demand' 'low_bike_demand']


In [66]:
# Model function

def model_function(X_train, Y_train, X_test, Y_test, numeric_vars, categorical_vars, binomial_vars, class_weight=None):

  #Scaling
  scaler = StandardScaler()
  num_var_train_scaled = scaler.fit_transform(X_train[numeric_vars])
  num_var_test_scaled = scaler.transform(X_test[numeric_vars])

  #Dummies
  cat_var_dum_train = pd.get_dummies(X_train[categorical_vars], drop_first=True)
  cat_var_dum_test  = pd.get_dummies(X_test[categorical_vars],  drop_first=True)

  X_train_scaled = np.hstack([num_var_train_scaled, cat_var_dum_train.values, X_train[binomial_vars].values])
  X_test_scaled  = np.hstack([num_var_test_scaled,  cat_var_dum_test.values,  X_test[binomial_vars].values])

  # Log Regr
  model = skl_lm.LogisticRegression(solver='lbfgs', max_iter=1000, class_weight=class_weight)
  model.fit(X_train_scaled, Y_train)

  # Evaluate on training data
  train_predict = model.predict(X_test_scaled)
  
  # Metrics
  accuracy = accuracy_score(Y_test, train_predict)

  confusion_mat = confusion_matrix(Y_test, train_predict)

  df_confusion_matrix = pd.DataFrame(confusion_mat,
                                    index=['Actual High_demand', 'Actual Low_demand'],
                                    columns=['Predicted High_demand', 'Predicted Low_demand'])

  precision = precision_score(Y_test, train_predict, pos_label='high_bike_demand')

  recall = recall_score(Y_test, train_predict, pos_label='high_bike_demand')

  f1 = f1_score(Y_test, train_predict, pos_label='high_bike_demand')

  # GridSearchCV
  param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
  grid = GridSearchCV(model, param_grid, cv=5, scoring='f1_macro')
  grid.fit(X_train_scaled, Y_train)
  
  return model, accuracy, df_confusion_matrix, precision, recall, f1, grid.best_params_, grid.best_score_, train_predict


In [67]:

X_train_copy = X_train.copy()
X_test_copy = X_test.copy()

In [68]:
# Try5 - temp, humidity, is Day, Good weather , class = balanced

X_train_copy['is_day'] = ((X_train_copy['hour_of_day'] >= 8) & (X_train_copy['hour_of_day'] <= 20)).astype(int)
X_test_copy['is_day']  = ((X_test_copy['hour_of_day'] >= 8) & (X_test_copy['hour_of_day'] <= 20)).astype(int)

X_train_copy['good_weather'] = ((X_train_copy['temp'] > 12) & (X_train_copy['humidity'] < 75)).astype(int)
X_test_copy['good_weather']  = ((X_test_copy['temp'] > 12) & (X_test_copy['humidity'] < 75)).astype(int)

model5, accuracy5, df_confusion_matrix5, precision5, recall5, f1_5, best_parameters5, best_score_5, train_predict5 = model_function(
    X_train_copy, Y_train, X_test_copy, Y_test,
    numeric_vars=['temp', 'humidity','good_weather'],
    categorical_vars=['is_day','day_of_week','month'],
    binomial_vars=['holiday','weekday','summertime','snow'],
    class_weight= 'balanced')


In [70]:
print("Accuracy:", accuracy5)
print(classification_report(Y_test, train_predict5))


Accuracy: 0.825
                  precision    recall  f1-score   support

high_bike_demand       0.52      0.92      0.67        61
 low_bike_demand       0.98      0.80      0.88       259

        accuracy                           0.82       320
       macro avg       0.75      0.86      0.77       320
    weighted avg       0.89      0.82      0.84       320

